In [7]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

# R0计算设置
R0_TARGET_AFTER_PAUSE_SEC = 0.5   # pause段最后一个测量点之后外推的R0时间
R0_FIT_POINT_START = 2            # 有效pulse起始点
R0_FIT_POINT_END = 6              # 有效pulse结束点
ZERO_CURRENT_LIMIT = 1e-6
VOLTAGE_JUMP_LIMIT = 1e-6

TIME_DIFF_OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current"
]

# R0及质量检验列
PULSE_OUTPUT_COLUMNS = TIME_DIFF_OUTPUT_COLUMNS + [
    "R0",
    "R0_Target_Time",
    "R0_Quality"
]


Note: you may need to restart the kernel to use updated packages.


In [8]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [9]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间、电流、电压处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")
    df_td["Voltage"] = pd.to_numeric(df_td["Voltage"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[TIME_DIFF_OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def get_effective_start_pos(group):
    # 返回pulse段内第一个非零有效电流点的位置。
        current_abs = group["Current"].abs().to_numpy()
        non_zero_pos = np.flatnonzero(current_abs > ZERO_CURRENT_LIMIT)

        if len(non_zero_pos) == 0:
            return None

        return int(non_zero_pos[0])

    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            return True

        # 如果首个测量点 Current == 0，则从第一个非零有效点开始判断稳定性
        current_values = group["Current"].iloc[effective_start_pos:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    pulse_sequence = pulse_sequence.groupby(["File", "pulse_segment_id"]).filter(
        lambda group: not is_bad_current_segment(group)
    ).copy()

    # -----------------------------
    # 7. 计算R0，并且每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def add_quality(base_quality, new_quality):
        if base_quality == "正常":
            return new_quality
        return base_quality + "；" + new_quality

    def calculate_r0_for_segment(group, effective_start_pos):
        group = group.sort_values("Time")

        r0_result = {
            "R0": np.nan,
            "R0_Target_Time": pd.NaT,
            "R0_Quality": "正常"
        }

        file_name = group["File"].iloc[0]
        pulse_start_time = group["Time"].iloc[0]
        first_current = group["Current"].iloc[0]

        previous_pause = df_td[
            (df_td["File"] == file_name)
            & (df_td["Time"] < pulse_start_time)
            & (df_td["Zustand"].str.startswith("PAU", na=False))
        ].sort_values("Time").tail(1)

        if previous_pause.empty:
            r0_result["R0_Quality"] = "无法计算R0：无前置pause点"
            return r0_result

        pause_time = previous_pause["Time"].iloc[0]
        pause_voltage = previous_pause["Voltage"].iloc[0]
        target_time = pause_time + pd.Timedelta(seconds=R0_TARGET_AFTER_PAUSE_SEC)
        r0_result["R0_Target_Time"] = target_time

        # pulse段第一个点为0时：无论voltage是否跳变，R0计算都从第一个非零有效点开始；
        # 若voltage已经跳变，则额外给质量标记。
        if abs(first_current) <= ZERO_CURRENT_LIMIT:
            first_voltage = group["Voltage"].iloc[0]

            if (
                pd.notna(first_voltage)
                and pd.notna(pause_voltage)
                and abs(first_voltage - pause_voltage) > VOLTAGE_JUMP_LIMIT
            ):
                r0_result["R0_Quality"] = "首点0且电压跳变"

        fit_start_pos = effective_start_pos + R0_FIT_POINT_START - 1
        fit_end_pos = effective_start_pos + R0_FIT_POINT_END

        fit_points = group.iloc[fit_start_pos:fit_end_pos].dropna(subset=["Time", "Voltage"])

        if len(fit_points) < 2:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：拟合点不足"
            )
            return r0_result

        effective_current = group["Current"].iloc[effective_start_pos]

        if abs(effective_current) <= ZERO_CURRENT_LIMIT:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：无非零有效电流"
            )
            return r0_result

        if pd.isna(pause_voltage):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：pause电压缺失"
            )
            return r0_result
        
        # 判断采样点距离pulse起点的距离
        x_sec = (fit_points["Time"] - target_time) / pd.Timedelta(seconds=1)

        y_voltage = fit_points["Voltage"].astype(float)

        slope, intercept = np.polyfit(x_sec.to_numpy(), y_voltage.to_numpy(), 1)
        extrapolated_voltage = intercept

        r0_result["R0"] = abs(
            (extrapolated_voltage - pause_voltage) / effective_current
        )

        return r0_result

    selected_indices = []
    r0_results = {}

    for _, group in pulse_sequence.groupby(["File", "pulse_segment_id"], sort=False):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            continue

        # 保留原逻辑：每段用“有效pulse起点后的第2个测量点”记录；
        # 如果点数不够，则退回到有效pulse起点本身。
        record_pos = effective_start_pos + 1
        if record_pos >= len(group):
            record_pos = effective_start_pos

        record_index = group.index[record_pos]
        selected_indices.append(record_index)

        r0_results[record_index] = calculate_r0_for_segment(
            group,
            effective_start_pos
        )

    pulse_sequence = pulse_sequence.loc[selected_indices].copy()

    for column in ["R0", "R0_Target_Time", "R0_Quality"]:
        pulse_sequence[column] = pulse_sequence.index.map(
            lambda idx: r0_results[idx][column]
        )

    pulse_sequence = pulse_sequence.reset_index(drop=True)

    # -----------------------------
    # 8. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[PULSE_OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence


pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)

# 统计 R0_Quality 各 flag 数量
if pulse_sequence.empty:
    print("pulse_sequence 为空，无法统计 R0_Quality")
else:
    r0_quality_flag_counts = (
        pulse_sequence["R0_Quality"]
        .fillna("缺失")
        .astype(str)
        .str.split("；")
        .explode()
        .str.strip()
        .value_counts()
        .rename_axis("R0_Quality_Flag")
        .reset_index(name="Count")
    )

    r0_quality_flag_counts["Percent_of_segments"] = (
        r0_quality_flag_counts["Count"] / len(pulse_sequence) * 100
    )

    display(r0_quality_flag_counts)

display(pulse_sequence)


,R0_Quality_Flag,Count,Percent_of_segments
0,正常,219,83.587786
1,无法计算R0：无前置pause点,24,9.160305
2,首点0且电压跳变,19,7.251908


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R0_Target_Time,R0_Quality
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 12:03:52.760000+00:00,-1.499470,4.036492,DCH,12_17,DCH/-1.5,NaN,NaT,无法计算R0：无前置pause点
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,0.025070,2024-11-12 13:04:35.250000+00:00,正常
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,0.024941,2024-11-12 14:05:18.390000+00:00,正常
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 19:55:56.640000+00:00,-1.498840,3.725950,DCH,12_35,DCH/-1.5,24.332409,2024-11-12 15:36:43.330000+00:00,正常
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,0.018330,2024-11-12 20:56:39.410000+00:00,正常
...,...,...,...,...,...,...,...,...,...,...,...,...
257,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,0.017858,2024-10-23 23:30:52.300000+00:00,正常
258,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 04:22:48.810000+00:00,-1.497851,3.192480,DCH,9_54,DCH/-1.5,42.808034,2024-10-24 00:01:13.750000+00:00,正常
259,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,0.019617,2024-10-24 05:23:31.420000+00:00,正常
260,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,0.019862,2024-10-24 06:24:14.600000+00:00,正常
